In [2]:
import pandas as pd
import numpy as np

In [3]:
transactions = pd.read_excel(
    "d:/MACHINE LEARNING PROJECT/data/raw/Dataset2_Transactions.xlsx"
)

In [4]:
transactions.head()

,Transaction_ID,Customer_ID,Brand,Purchase_Amount,Cashback,Payment_Method
0,T200000,C111606,PVR,547,166,Credit Card
1,T200001,C110791,Ajio,2789,301,Net Banking
2,T200002,C118737,MakeMyTrip,2376,171,Credit Card
3,T200003,C110608,Flipkart,3104,439,Debit Card
4,T200004,C117948,Myntra,1283,481,Net Banking


In [5]:
print("Shape:", transactions.shape)

Shape: (25000, 6)


In [6]:
print(transactions.columns.tolist())

['Transaction_ID', 'Customer_ID', 'Brand', 'Purchase_Amount', 'Cashback', 'Payment_Method']


In [7]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Transaction_ID   25000 non-null  object
 1   Customer_ID      25000 non-null  object
 2   Brand            25000 non-null  object
 3   Purchase_Amount  25000 non-null  int64 
 4   Cashback         25000 non-null  int64 
 5   Payment_Method   25000 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.1+ MB


In [8]:
transactions.isnull().sum()

Transaction_ID     0
Customer_ID        0
Brand              0
Purchase_Amount    0
Cashback           0
Payment_Method     0
dtype: int64

In [9]:
transactions.describe()

,Purchase_Amount,Cashback
count,25000.000000,25000.000000
mean,5035.535080,250.148480
std,2858.185508,144.023451
min,101.000000,1.000000
25%,2570.000000,125.000000
50%,5030.000000,250.000000
75%,7505.000000,375.000000
max,9999.000000,499.000000


In [10]:
print(transactions['Payment_Method'].unique())

['Credit Card' 'Net Banking' 'Debit Card' 'UPI']


In [11]:
print(transactions['Purchase_Amount'].describe())

count    25000.000000
mean      5035.535080
std       2858.185508
min        101.000000
25%       2570.000000
50%       5030.000000
75%       7505.000000
max       9999.000000
Name: Purchase_Amount, dtype: float64


In [12]:
transactions['Purchase_Amount'] = pd.to_numeric(
    transactions['Purchase_Amount'],
    errors='coerce'
)

In [13]:
customer_transaction_count = transactions.groupby(
    'Customer_ID'
)['Transaction_ID'].transform('count')

transactions['Transaction_Count'] = customer_transaction_count

In [14]:
transactions[
    [
        'Customer_ID',
        'Transaction_ID',
        'Purchase_Amount',
        'Transaction_Count'
    ]
].head(10)

,Customer_ID,Transaction_ID,Purchase_Amount,Transaction_Count
0,C111606,T200000,547,2
1,C110791,T200001,2789,2
2,C118737,T200002,2376,2
3,C110608,T200003,3104,1
4,C117948,T200004,1283,2
5,C114921,T200005,1465,3
6,C123261,T200006,8584,3
7,C106841,T200007,3538,1
8,C113534,T200008,1098,1
9,C102345,T200009,8640,4


In [15]:
transactions[
    [
        'Purchase_Amount',
        'Transaction_Count'
    ]
].describe()

,Purchase_Amount,Transaction_Count
count,25000.000000,25000.000000
mean,5035.535080,1.994400
std,2858.185508,0.996358
min,101.000000,1.000000
25%,2570.000000,1.000000
50%,5030.000000,2.000000
75%,7505.000000,3.000000
max,9999.000000,7.000000


In [16]:
fraud_threshold = transactions['Purchase_Amount'].quantile(0.95)

print("Fraud Threshold:", fraud_threshold)

Fraud Threshold: 9496.05


In [17]:
transactions['Fraud'] = np.where(
    transactions['Purchase_Amount'] >= fraud_threshold,
    1,
    0
)

In [18]:
print(transactions['Fraud'].value_counts())

Fraud
0    23750
1     1250
Name: count, dtype: int64


In [19]:
print(
    transactions['Fraud'].value_counts(normalize=True) * 100
)

Fraud
0    95.0
1     5.0
Name: proportion, dtype: float64


In [20]:
transactions[
    ['Transaction_ID', 'Purchase_Amount', 'Fraud']
].head(10)

,Transaction_ID,Purchase_Amount,Fraud
0,T200000,547,0
1,T200001,2789,0
2,T200002,2376,0
3,T200003,3104,0
4,T200004,1283,0
5,T200005,1465,0
6,T200006,8584,0
7,T200007,3538,0
8,T200008,1098,0
9,T200009,8640,0


In [21]:
fraud_features = [
    'Purchase_Amount',
    'Transaction_Count'
]

In [22]:
X = transactions[fraud_features]
y = transactions['Fraud']

In [23]:
print(X.isnull().sum())

Purchase_Amount      0
Transaction_Count    0
dtype: int64


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
fraud_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [27]:
fraud_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [28]:
y_pred = fraud_model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Fraud Model Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Fraud Model Accuracy: 1.0
Accuracy Percentage: 100.0 %


In [30]:
transactions.to_csv(
    "../data/processed/fraud_data.csv",
    index=False
)

In [31]:
import joblib

In [32]:
joblib.dump(
    fraud_model,
    "../models/fraud_model.pkl"
)

['../models/fraud_model.pkl']